# LAB 5 — Contextual Retrieval (#T09)
## Aula 4: OpenSearch Completo — Dense, Hybrid Search, Neural Sparse e Contextual Retrieval
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

---

**Duração estimada:** 45 minutos  
**Técnica:** #T09 Contextual Retrieval  
**Pré-requisitos:** LAB1 (índice híbrido criado), LAB4 (Neural Sparse funcionando), vLLM rodando

---

## Objetivo

Neste laboratório vamos:

1. Carregar 100 chunks do corpus jurídico
2. Medir a **Context Precision (RAGAS) baseline** — sem contextualização
3. Enriquecer cada chunk com um **resumo contextual gerado pelo vLLM** (Llama 3.1 8B)
4. Reindexar os chunks enriquecidos no OpenSearch 3.x
5. Medir a **Context Precision (RAGAS) após** o Contextual Retrieval
6. Comparar e registrar os resultados no **LangFuse**

---

## Conceito: Por que Chunks sem Contexto Falham?

```
Chunk isolado:
  "O Tribunal, por unanimidade, negou provimento ao recurso."
  → Embedding genérico, retrieval fraco

Chunk enriquecido com Contextual Retrieval:
  "[CONTEXTO: Acórdão STJ REsp 1234567/SP, crime ambiental —
   desmatamento em APP. Dispositivo final do voto do relator.]
   O Tribunal, por unanimidade, negou provimento ao recurso."
  → Embedding rico, retrieval preciso
```

**Referência:** ANTHROPIC. *Introducing Contextual Retrieval*. 2024.

---
## Passo 1 — Instalação de Dependências

In [ ]:
# Instalar dependências do LAB5
!pip install -q opensearch-py langchain langchain-openai langchain-community \
               sentence-transformers ragas datasets pandas tqdm langfuse

print("✅ Dependências instaladas")

---
## Passo 2 — Configuração de Variáveis de Ambiente

In [ ]:
import os

# ── OpenSearch 3.x ──────────────────────────────────────────────
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
OPENSEARCH_USER = os.getenv("OPENSEARCH_USER", "admin")
OPENSEARCH_PASS = os.getenv("OPENSEARCH_PASS", "admin")

# Índices
INDEX_BASELINE    = "juridico_baseline_aula4"        # chunks sem contexto (do LAB1)
INDEX_CONTEXTUAL  = "juridico_contextual_aula4"      # chunks enriquecidos (novo)

# ── vLLM ────────────────────────────────────────────────────────
VLLM_HOST = os.getenv("VLLM_HOST", "localhost")
VLLM_PORT = int(os.getenv("VLLM_PORT", "8000"))
VLLM_BASE_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
VLLM_MODEL    = "meta-llama/Llama-3.1-8B-Instruct"

# ── LangFuse ────────────────────────────────────────────────────
LANGFUSE_PUBLIC_KEY  = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-xxxx")
LANGFUSE_SECRET_KEY  = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-xxxx")
LANGFUSE_HOST        = os.getenv("LANGFUSE_HOST", "http://localhost:3000")

# ── Embedding ───────────────────────────────────────────────────
EMBEDDING_MODEL = "BAAI/bge-m3"
EMBEDDING_DIM   = 1024

print(f"OpenSearch: {OPENSEARCH_HOST}:{OPENSEARCH_PORT}")
print(f"vLLM: {VLLM_BASE_URL}")
print(f"LangFuse: {LANGFUSE_HOST}")
print(f"Índice baseline : {INDEX_BASELINE}")
print(f"Índice contextual: {INDEX_CONTEXTUAL}")

---
## Passo 3 — Conexão com OpenSearch 3.x e vLLM

In [ ]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from langchain_openai import ChatOpenAI
from sentence_transformers import SentenceTransformer
import json

# Conexão OpenSearch 3.x
os_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False,
    verify_certs=False,
    connection_class=RequestsHttpConnection,
)

info = os_client.info()
print(f"✅ OpenSearch conectado — versão: {info['version']['number']}")
assert info['version']['number'].startswith('3'), "⚠️ ATENÇÃO: este lab requer OpenSearch 3.x!"

# vLLM
llm = ChatOpenAI(
    model=VLLM_MODEL,
    base_url=VLLM_BASE_URL,
    api_key="dummy",
    temperature=0.0,
    max_tokens=200,
)

# Teste rápido vLLM
resp = llm.invoke("Responda em uma palavra: qual é a capital do Brasil?")
print(f"✅ vLLM respondeu: {resp.content.strip()}")

# Modelo de embedding
embedder = SentenceTransformer(EMBEDDING_MODEL)
print(f"✅ Embedding model carregado: {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")

---
## Passo 4 — Carregar Corpus Jurídico e Criar Chunks

In [ ]:
import json
import re
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Carregar corpus da Aula 4
CORPUS_PATH = "../datasets/corpus_juridico_aula4.json"

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Documentos carregados: {len(corpus)}")
for doc in corpus[:3]:
    print(f"  - [{doc.get('tipo','?')}] {doc.get('id','?')}: {len(doc.get('conteudo',''))} chars")

# Chunking: recursive character splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " "],
)

# Gerar até 100 chunks para o lab
chunks_raw = []
for doc in corpus:
    partes = splitter.split_text(doc.get("conteudo", ""))
    for i, parte in enumerate(partes):
        chunks_raw.append({
            "chunk_id": f"{doc['id']}_chunk_{i:03d}",
            "doc_id": doc["id"],
            "doc_tipo": doc.get("tipo", "desconhecido"),
            "doc_completo": doc["conteudo"][:3000],  # primeiros 3000 chars para contexto
            "conteudo": parte,
            "indice": i,
        })
        if len(chunks_raw) >= 100:
            break
    if len(chunks_raw) >= 100:
        break

print(f"\n✅ Total de chunks gerados: {len(chunks_raw)}")
print(f"\nExemplo de chunk sem contexto:")
print(f"  ID: {chunks_raw[0]['chunk_id']}")
print(f"  Conteúdo: {chunks_raw[0]['conteudo'][:150]}...")

---
## Passo 5 — Criação do Índice Baseline (sem Contextual Retrieval)

Primeiro indexamos os chunks **sem enriquecimento** para medir o RAGAS baseline.

In [ ]:
import numpy as np
from tqdm.notebook import tqdm

# ── Mapeamento do índice OpenSearch 3.x ──────────────────────────
MAPPING_JURIDICO = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 100,
        }
    },
    "mappings": {
        "properties": {
            "chunk_id":    {"type": "keyword"},
            "doc_id":      {"type": "keyword"},
            "doc_tipo":    {"type": "keyword"},
            "conteudo":    {"type": "text", "analyzer": "portuguese"},
            "embedding":   {
                "type": "knn_vector",
                "dimension": EMBEDDING_DIM,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "faiss",
                    "parameters": {"ef_construction": 128, "m": 16}
                }
            },
        }
    }
}

def criar_indice(nome: str, recriar: bool = True):
    if recriar and os_client.indices.exists(index=nome):
        os_client.indices.delete(index=nome)
        print(f"  🗑️ Índice {nome} removido")
    os_client.indices.create(index=nome, body=MAPPING_JURIDICO)
    print(f"  ✅ Índice {nome} criado")

def indexar_chunks(nome_indice: str, chunks: list, campo_conteudo: str = "conteudo"):
    """Indexa lista de chunks gerando embeddings em batch."""
    print(f"Gerando embeddings para {len(chunks)} chunks...")
    textos = [c[campo_conteudo] for c in chunks]
    embeddings = embedder.encode(textos, batch_size=16, show_progress_bar=True)

    print(f"Indexando no OpenSearch 3.x...")
    bulk_body = []
    for chunk, emb in tqdm(zip(chunks, embeddings), total=len(chunks)):
        bulk_body.append({"index": {"_index": nome_indice, "_id": chunk["chunk_id"]}})
        bulk_body.append({
            "chunk_id": chunk["chunk_id"],
            "doc_id":   chunk["doc_id"],
            "doc_tipo": chunk["doc_tipo"],
            "conteudo": chunk[campo_conteudo],
            "embedding": emb.tolist(),
        })

    resp = os_client.bulk(body=bulk_body)
    erros = [i for i in resp["items"] if i["index"].get("error")]
    print(f"✅ Indexados: {len(chunks) - len(erros)} | Erros: {len(erros)}")
    os_client.indices.refresh(index=nome_indice)

# Criar e indexar baseline
print("=== CRIANDO ÍNDICE BASELINE ===")
criar_indice(INDEX_BASELINE)
indexar_chunks(INDEX_BASELINE, chunks_raw, campo_conteudo="conteudo")
print(f"\nTotal de docs no índice baseline: {os_client.count(index=INDEX_BASELINE)['count']}")

---
## Passo 6 — Enriquecimento com Contextual Retrieval via vLLM

Para cada chunk, enviamos ao vLLM um prompt com o documento completo e o trecho,
pedindo uma frase de contexto situacional. O chunk enriquecido = contexto + chunk original.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import asyncio
import time

PROMPT_CONTEXTO = ChatPromptTemplate.from_messages([
    ("system",
     "Você é um analista jurídico especializado no direito brasileiro. "
     "Sua tarefa é gerar uma frase de contexto curta e precisa."),
    ("human",
     "Documento jurídico completo (início):\n{documento}\n\n"
     "Trecho a contextualizar:\n{chunk}\n\n"
     "Gere UMA frase (máximo 40 palavras) descrevendo o papel deste trecho no documento. "
     "Inclua: tipo do documento, partes/números relevantes, assunto e posição no texto. "
     "Responda SOMENTE a frase, sem introduções ou explicações."
    )
])

def gerar_contexto_sincrono(documento: str, chunk: str) -> str:
    """Gera contexto situacional para um chunk via vLLM (chamada síncrona)."""
    try:
        resp = llm.invoke(
            PROMPT_CONTEXTO.format_messages(documento=documento, chunk=chunk)
        )
        return resp.content.strip()
    except Exception as e:
        print(f"    ⚠️ Erro na geração de contexto: {e}")
        return "[Contexto não disponível]"

def enriquecer_chunk(chunk: dict) -> dict:
    """Retorna chunk com campo 'conteudo_contextual' adicionado."""
    contexto = gerar_contexto_sincrono(
        documento=chunk["doc_completo"],
        chunk=chunk["conteudo"]
    )
    chunk_enriquecido = chunk.copy()
    chunk_enriquecido["contexto_gerado"] = contexto
    chunk_enriquecido["conteudo_contextual"] = (
        f"[CONTEXTO: {contexto}]\n\n{chunk['conteudo']}"
    )
    return chunk_enriquecido

# Enriquecer os 100 chunks (com barra de progresso)
print("=== ENRIQUECENDO CHUNKS COM CONTEXTUAL RETRIEVAL ===")
print(f"Processando {len(chunks_raw)} chunks via vLLM ({VLLM_MODEL})...")
print("Estimativa: ~30–60s no vLLM local com GPU T4\n")

chunks_enriquecidos = []
inicio = time.time()

for i, chunk in enumerate(tqdm(chunks_raw)):
    chunk_enc = enriquecer_chunk(chunk)
    chunks_enriquecidos.append(chunk_enc)
    if (i + 1) % 10 == 0:
        elapsed = time.time() - inicio
        print(f"  {i+1}/100 chunks | Tempo: {elapsed:.0f}s | Ex: '{chunk_enc['contexto_gerado'][:80]}...'")

total_time = time.time() - inicio
print(f"\n✅ Enriquecimento concluído em {total_time:.1f}s")
print(f"\nExemplo de chunk enriquecido:")
print(f"  Original : {chunks_enriquecidos[0]['conteudo'][:100]}...")
print(f"  Contexto : {chunks_enriquecidos[0]['contexto_gerado']}")
print(f"  Enriquecido: {chunks_enriquecidos[0]['conteudo_contextual'][:200]}...")

---
## Passo 7 — Indexar Chunks Enriquecidos no OpenSearch 3.x

In [ ]:
print("=== CRIANDO ÍNDICE CONTEXTUAL ===")
criar_indice(INDEX_CONTEXTUAL)
indexar_chunks(INDEX_CONTEXTUAL, chunks_enriquecidos, campo_conteudo="conteudo_contextual")
print(f"\nTotal de docs no índice contextual: {os_client.count(index=INDEX_CONTEXTUAL)['count']}")

# Verificar estrutura de um documento no índice
sample = os_client.get(index=INDEX_CONTEXTUAL, id=chunks_enriquecidos[0]["chunk_id"])
print(f"\nAmostra do índice contextual:")
print(f"  chunk_id: {sample['_source']['chunk_id']}")
print(f"  conteudo: {sample['_source']['conteudo'][:200]}...")
print(f"  embedding shape: {len(sample['_source']['embedding'])} dims")

---
## Passo 8 — Avaliação com RAGAS: Context Precision Antes vs. Depois

Usamos 5 queries de avaliação do dataset da Aula 4 para medir Context Precision.

In [ ]:
import numpy as np
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall

# Queries de avaliação (5 queries com ground-truth do dataset)
QUERIES_AVALIACAO = [
    {
        "question": "Quais são os requisitos para decretação de prisão preventiva?",
        "ground_truth": "A prisão preventiva pode ser decretada pelo juiz mediante representação da autoridade policial ou requerimento do Ministério Público, quando presentes os requisitos do art. 312 do CPP: garantia da ordem pública, da ordem econômica, por conveniência da instrução criminal, ou para assegurar a aplicação da lei penal."
    },
    {
        "question": "Como se caracteriza o crime ambiental de desmatamento em APP?",
        "ground_truth": "O desmatamento em Área de Preservação Permanente configura crime ambiental previsto na Lei 9.605/1998, punível com detenção e/ou multa, independentemente do bioma afetado."
    },
    {
        "question": "Quais os efeitos da reincidência no cálculo da pena?",
        "ground_truth": "A reincidência é circunstância agravante que aumenta a pena no momento da fixação da pena definitiva, conforme art. 61, I do Código Penal, podendo ainda impedir o benefício da suspensão condicional da pena."
    },
    {
        "question": "O que é habeas corpus e quando pode ser impetrado?",
        "ground_truth": "O habeas corpus é ação constitucional que protege o direito de locomoção, cabível sempre que alguém sofrer ou se achar ameaçado de sofrer violência ou coação em sua liberdade de locomoção por ilegalidade ou abuso de poder (art. 5º, LXVIII, CF/88)."
    },
    {
        "question": "Quais documentos compõem o boletim de ocorrência policial?",
        "ground_truth": "O boletim de ocorrência registra: qualificação das partes, relato dos fatos, tipificação criminal, relação de testemunhas, medidas cautelares adotadas e assinatura da autoridade policial responsável."
    },
]

def buscar_contextos(query: str, indice: str, k: int = 5) -> list[str]:
    """Executa busca híbrida e retorna top-k contextos."""
    query_emb = embedder.encode([query])[0].tolist()

    resp = os_client.search(
        index=indice,
        body={
            "size": k,
            "query": {
                "hybrid": {
                    "queries": [
                        {"match": {"conteudo": {"query": query}}},
                        {"knn": {"embedding": {"vector": query_emb, "k": k}}}
                    ]
                }
            }
        }
    )
    return [hit["_source"]["conteudo"] for hit in resp["hits"]["hits"]]

# Montar datasets para RAGAS
dados_baseline    = []
dados_contextual  = []

print("Executando buscas para avaliação RAGAS...")
for q in QUERIES_AVALIACAO:
    ctx_baseline   = buscar_contextos(q["question"], INDEX_BASELINE)
    ctx_contextual = buscar_contextos(q["question"], INDEX_CONTEXTUAL)

    dados_baseline.append({
        "question":     q["question"],
        "contexts":     ctx_baseline,
        "ground_truth": q["ground_truth"],
        "answer":       "[avaliação de retrieval — sem geração]"
    })
    dados_contextual.append({
        "question":     q["question"],
        "contexts":     ctx_contextual,
        "ground_truth": q["ground_truth"],
        "answer":       "[avaliação de retrieval — sem geração]"
    })
    print(f"  ✅ '{q['question'][:50]}...' — {len(ctx_baseline)} ctx baseline, {len(ctx_contextual)} ctx contextual")

print("\nAvaliando com RAGAS...")
resultados_baseline   = evaluate(Dataset.from_list(dados_baseline),   metrics=[context_precision, context_recall])
resultados_contextual = evaluate(Dataset.from_list(dados_contextual), metrics=[context_precision, context_recall])

print("\n" + "="*60)
print(" RESULTADOS RAGAS — BASELINE vs CONTEXTUAL RETRIEVAL")
print("="*60)
print(f"  Context Precision — Baseline  : {resultados_baseline['context_precision']:.4f}")
print(f"  Context Precision — Contextual: {resultados_contextual['context_precision']:.4f}")
delta_cp = resultados_contextual['context_precision'] - resultados_baseline['context_precision']
print(f"  Delta Context Precision       : {delta_cp:+.4f} ({delta_cp*100:+.1f}%)")
print()
print(f"  Context Recall    — Baseline  : {resultados_baseline['context_recall']:.4f}")
print(f"  Context Recall    — Contextual: {resultados_contextual['context_recall']:.4f}")
delta_cr = resultados_contextual['context_recall'] - resultados_baseline['context_recall']
print(f"  Delta Context Recall          : {delta_cr:+.4f} ({delta_cr*100:+.1f}%)")
print("="*60)

---
## Passo 9 — Registrar Resultados no LangFuse

In [ ]:
from langfuse import Langfuse
import datetime

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST,
)

# Trace principal para o experimento
trace = langfuse.trace(
    name="Aula4-LAB5-Contextual-Retrieval",
    metadata={
        "aula": "4",
        "lab": "5",
        "tecnica": "#T09 Contextual Retrieval",
        "n_chunks": len(chunks_raw),
        "embedding_model": EMBEDDING_MODEL,
        "llm_model": VLLM_MODEL,
        "opensearch_version": "3.x",
        "data": datetime.datetime.now().isoformat(),
    }
)

# Span do experimento baseline
span_baseline = trace.span(
    name="ragas-baseline",
    metadata={"indice": INDEX_BASELINE, "tipo": "chunks sem contexto"}
)
span_baseline.score(
    name="context_precision",
    value=float(resultados_baseline['context_precision'])
)
span_baseline.score(
    name="context_recall",
    value=float(resultados_baseline['context_recall'])
)
span_baseline.end()

# Span do experimento com Contextual Retrieval
span_contextual = trace.span(
    name="ragas-contextual-retrieval",
    metadata={
        "indice": INDEX_CONTEXTUAL,
        "tipo": "chunks com contexto vLLM",
        "tempo_enriquecimento_s": round(total_time, 1)
    }
)
span_contextual.score(
    name="context_precision",
    value=float(resultados_contextual['context_precision'])
)
span_contextual.score(
    name="context_recall",
    value=float(resultados_contextual['context_recall'])
)
span_contextual.end()

# Score do delta (melhoria)
trace.score(
    name="delta_context_precision",
    value=float(delta_cp),
    comment=f"Melhoria absoluta em Context Precision com Contextual Retrieval"
)

langfuse.flush()
print(f"✅ Resultado registrado no LangFuse")
print(f"   Trace URL: {LANGFUSE_HOST}/traces/{trace.id}")

---
## Passo 10 — Análise Qualitativa: Comparando Contextos Recuperados

In [ ]:
import pandas as pd

print("=== ANÁLISE QUALITATIVA — EXEMPLO DE RECUPERAÇÃO ===")
query_exemplo = QUERIES_AVALIACAO[0]["question"]
print(f"Query: '{query_exemplo}'\n")

ctx_b = buscar_contextos(query_exemplo, INDEX_BASELINE, k=3)
ctx_c = buscar_contextos(query_exemplo, INDEX_CONTEXTUAL, k=3)

print("── BASELINE (sem contexto) ──")
for i, ctx in enumerate(ctx_b, 1):
    print(f"  [{i}] {ctx[:200]}...\n")

print("── CONTEXTUAL RETRIEVAL (com contexto vLLM) ──")
for i, ctx in enumerate(ctx_c, 1):
    print(f"  [{i}] {ctx[:200]}...\n")

# Tabela-resumo dos resultados
df_resultados = pd.DataFrame([
    {
        "Estratégia": "Baseline (sem contexto)",
        "Context Precision": f"{resultados_baseline['context_precision']:.4f}",
        "Context Recall":    f"{resultados_baseline['context_recall']:.4f}",
        "Tempo ingestão": "~2s / 100 chunks",
    },
    {
        "Estratégia": "Contextual Retrieval (#T09)",
        "Context Precision": f"{resultados_contextual['context_precision']:.4f}",
        "Context Recall":    f"{resultados_contextual['context_recall']:.4f}",
        "Tempo ingestão": f"~{total_time:.0f}s / 100 chunks",
    },
])

print("\n" + "="*60)
print(" TABELA COMPARATIVA FINAL")
print("="*60)
print(df_resultados.to_string(index=False))

# Exportar CSV
df_resultados.to_csv("resultados_contextual_retrieval_lab5.csv", index=False)
print("\n✅ Resultados exportados: resultados_contextual_retrieval_lab5.csv")

---
## ✅ Checklist de Conclusão do LAB5

Antes de prosseguir para o LAB6, verifique todos os itens:

| # | Item | Status |
|---|------|--------|
| 1 | OpenSearch 3.x conectado e versão confirmada | ☐ |
| 2 | vLLM respondendo à chamada de teste | ☐ |
| 3 | 100 chunks gerados e indexados no índice baseline | ☐ |
| 4 | Enriquecimento contextual concluído (contexto_gerado preenchido) | ☐ |
| 5 | Índice contextual criado e indexado com conteudo_contextual | ☐ |
| 6 | RAGAS executado — Context Precision e Recall medidos nos dois índices | ☐ |
| 7 | Delta Context Precision ≥ 0 (contextual melhorou o baseline) | ☐ |
| 8 | Resultados registrados no LangFuse com trace visível | ☐ |
| 9 | CSV exportado com tabela comparativa | ☐ |

---

## Próximo Passo

→ **LAB6 — LangFuse Dashboard Comparativo**: visualizar e comparar latência e qualidade entre BM25, dense e hybrid nos dashboards do LangFuse.

---

## Referências

ANTHROPIC. **Introducing Contextual Retrieval**. Blog Anthropic, set. 2024. Disponível em: <https://www.anthropic.com/news/contextual-retrieval>.

ES, S. et al. **RAGAS: Automated Evaluation of Retrieval Augmented Generation**. arXiv:2309.15217, 2023.

OPENSEARCH PROJECT. **Hybrid Search**. OpenSearch 3.x Docs. Disponível em: <https://docs.opensearch.org/latest/vector-search/ai-search/hybrid-search/>.